In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
import joblib

# تحميل البيانات
df = pd.read_csv("Final_Updates.csv")

# تحويل عمود 'Datetime' إلى نوع بيانات datetime
df['Datetime'] = pd.to_datetime(df['Datetime'])

# استخراج الميزات الزمنية
df['Month'] = df['Datetime'].dt.month
df['Day'] = df['Datetime'].dt.day
df['Hour'] = df['Datetime'].dt.hour
df['Minute'] = df['Datetime'].dt.minute
df['Weekday'] = df['Datetime'].dt.day_name()

# ترميز الأعمدة الفئوية
label_encoder_city = LabelEncoder()
df['City'] = label_encoder_city.fit_transform(df['City'])

label_encoder_congestion = LabelEncoder()
df['CongestionLevel'] = label_encoder_congestion.fit_transform(df['CongestionLevel'])

label_encoder_weekday = LabelEncoder()
df['Weekday'] = label_encoder_weekday.fit_transform(df['Weekday'])

# تحديث قائمة الميزات لتشمل الميزات الجديدة
features = [
    'TrafficIndexLive_norm', 'JamsCount_norm', 'TrafficIndexWeekAgo_norm',
    'TravelTimeHistoric_norm', 'TravelTimeLive_norm',
    'Hour', 'Weekday', 'IsWeekend', 'Year', 'Month', 'Day', 'Minute'
]

# اختيار الميزات والهدف
X = df[features]
y = df['CongestionLevel']

In [2]:
# تقسيم البيانات إلى مجموعة تدريب واختبار (60% تدريب، 40% اختبار)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# تطبيع البيانات باستخدام StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# تدريب نموذج SVM
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_scaled, y_train)

# حفظ النموذج
joblib.dump(svm_model, 'svm_congestion_model.pkl')

# حساب التنبؤات
y_pred = svm_model.predict(X_test_scaled)

# حساب الدقة
accuracy = accuracy_score(y_test, y_pred)

# تحديد أسماء الفئات النصية
target_names = label_encoder_congestion.classes_

# طباعة تقرير التصنيف
classification_report_result = classification_report(y_test, y_pred, target_names=target_names.astype(str))  # تحويل القيم إلى نصوص

print(f"Model Accuracy: {accuracy}")
print("Classification Report:")
print(classification_report_result)

Model Accuracy: 0.9167701863354037
Classification Report:
              precision    recall  f1-score   support

        High       0.99      0.97      0.98      4035
         Low       0.00      0.00      0.00       239
    Moderate       0.59      0.90      0.71       556

    accuracy                           0.92      4830
   macro avg       0.53      0.62      0.56      4830
weighted avg       0.89      0.92      0.90      4830



C:\Users\Khaled-PC\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Khaled-PC\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Khaled-PC\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [3]:
# إدخال المستخدم
def get_user_input():
    return {
        'TrafficIndexLive': float(input("TrafficIndexLive: ")),
        'JamsCount': int(input("JamsCount: ")),
        'TrafficIndexWeekAgo': float(input("TrafficIndexWeekAgo: ")),
        'TravelTimeHistoric': float(input("TravelTimeHistoric: ")),
        'TravelTimeLive': float(input("TravelTimeLive: ")),
        'Hour': int(input("Hour (0-23): ")),
        'Weekday': int(input("Weekday (0=Monday to 6=Sunday): ")),
        'IsWeekend': int(input("IsWeekend (0 or 1): ")),
        'Year': int(input("Year: ")),
        'Month': int(input("Month: ")),
        'Day': int(input("Day: ")),
        'Minute': int(input("Minute: "))
    }



    return {
        'TrafficIndexLive': traffic_index_live,
        'JamsCount': jams_count,
        'TrafficIndexWeekAgo': traffic_index_week_ago,
        'TravelTimeHistoric': travel_time_historic,
        'TravelTimeLive': travel_time_live,
        'Hour': hour,
        'Weekday': weekday,
        'IsWeekend': is_weekend,
        'Year': year,
        'Month': month,
        'Day': day,
        'Minute': minute
    }


In [4]:
def normalize_input(user_input):
    user_input['TrafficIndexLive_norm'] = user_input['TrafficIndexLive'] / 99
    user_input['JamsCount_norm'] = user_input['JamsCount'] / 883
    user_input['TrafficIndexWeekAgo_norm'] = user_input['TrafficIndexWeekAgo'] / 99
    user_input['TravelTimeHistoric_norm'] = user_input['TravelTimeHistoric'] / 94.7783212767502
    user_input['TravelTimeLive_norm'] = user_input['TravelTimeLive'] / 130.973103722656
    return user_input


In [ ]:

# تحميل النموذج المدرب
svm_model = joblib.load('svm_congestion_model.pkl')

# دالة التنبؤ
def predict_congestion(user_input):
    user_input = normalize_input(user_input)

    # اختيار فقط الميزات المستخدمة في التدريب
    input_filtered = {key: user_input[key] for key in trained_features}
    user_input_df = pd.DataFrame([input_filtered], columns=trained_features)

    # تطبيع المدخلات باستخدام نفس الـ Scaler الذي استخدمناه في التدريب
    user_input_scaled = scaler.transform(user_input_df)

    # التنبؤ
    prediction = svm_model.predict(user_input_scaled)
    predicted_label = label_encoder_congestion.inverse_transform(prediction)
    return predicted_label[0]

# الاستخدام
user_input = get_user_input()
result = predict_congestion(user_input)
print(f"\nمستوى الزحمة المتوقع: {result}")